# Task 6: Fused Outer Multiplication + ReLU — Backward

Bot task id: `06_outer_mul_relu_block_backward`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/06_outer_mul_relu_block_backward.ipynb)

## Условие

Градиент по `x` для операции `z = relu(x[None, :] * y[:, None])`:

$$dx[j] = \sum_i \mathbb{1}[x[j] \cdot y[i] > 0] \cdot y[i] \cdot dz[i, j]$$

где `dz` — апстрим-градиент размера `(N1, N0)`.

**Требования**
- двумерная сетка `(pid1, pid0)`;
- размеры блоков `B0 < N0`, `B1 < N1`, маскирование на обеих осях;
- использовать `tl.arange`, `tl.load`, `tl.store`, `tl.where`;
- накапливать частичные суммы и писать в `dx` атомарным `tl.atomic_add`.

**Подсказка**
- `offs0` — столбцы (ось x), `offs1` — строки (ось y); загрузите куски `x`, `y`, `dz`;
- `u = y[:, None] * x[None, :]`, маска ReLU `relu_mask = u > 0`;
- `g = tl.where(relu_mask, y[:, None] * dz_tile, 0)`;
- просуммируйте по оси `i` (строки) и прибавьте к `dx[offs0]` через `tl.atomic_add`.

Раз программы складывают в `dx` атомарно, `dx` в обёртке должен начинаться с нулей.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `06_outer_mul_relu_block_backward.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 06_outer_mul_relu_block_backward.py
import torch
import triton
import triton.language as tl


@triton.jit
def _mul_relu_block_back_kernel(x_ptr, y_ptr, dz_ptr, dx_ptr, N0, N1, B0: tl.constexpr, B1: tl.constexpr):
    # YOUR CODE HERE
    pass


def mul_relu_block_back_triton(x: torch.Tensor, y: torch.Tensor, dz: torch.Tensor) -> torch.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "06_outer_mul_relu_block_backward.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x = torch.randn(100, device="cuda", requires_grad=True)
y = torch.randn(90, device="cuda")
dz = torch.randn(90, 100, device="cuda")
torch.relu(x[None, :] * y[:, None]).backward(dz)
torch.testing.assert_close(sol.mul_relu_block_back_triton(x.detach(), y, dz), x.grad, atol=1e-4, rtol=1e-4)
assert "tl.atomic_add" in SRC and "tl.where" in SRC, "the task asks for tl.where and tl.atomic_add"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("06_outer_mul_relu_block_backward.py")